<a href="https://colab.research.google.com/github/santidrelew/tp-mpg/blob/main/notebooks/02_extraccion_muestra.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **2 - Extracción de la muestra**

In [13]:
try:
    import google.colab
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB:
    !git clone https://github.com/santidrelew/tp-mpg.git
    %cd tp-mpg

Cloning into 'tp-mpg'...
remote: Enumerating objects: 51, done.
remote: Counting objects: 100% (51/51), done.
remote: Compressing objects: 100% (46/46), done.
remote: Total 51 (delta 20), reused 8 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (51/51), 4.54 MiB | 10.51 MiB/s, done.
Resolving deltas: 100% (20/20), done.
/content/tp-mpg/tp-mpg/tp-mpg


In [14]:
from pathlib import Path
import pandas as pd
import numpy as np
from scipy import stats

In [15]:
# Cargamos los datasets
PROJECT_DIR = Path().resolve() if EN_COLAB else Path().resolve().parent
DATA_BASE_DIR = PROJECT_DIR / "data" / "base"

df = pd.read_csv(DATA_BASE_DIR / "milan_airbnb_completo.csv")

Vamos a hacer un muestreo estratificado tomando como estrato la variable neighbourhood_cleansed. Esto es porque las tarifas y amenidades difieren sustancialmente entre el centro historico y las comunas perifericas. Un muestreo puramente aleatorio corria el riesgo de subestimar zonas de menor densidad de oferta (como la comuna 7).

In [ ]:
N_MUESTRA = 750
fraccion = N_MUESTRA / len(df)

df_muestra = pd.concat([
    grupo.sample(frac=fraccion, random_state=42)
    for _, grupo in df.groupby("neighbourhood_cleansed", sort=True)
])

# Redondeamos
if len(df_muestra) > N_MUESTRA:
  df_muestra = df_muestra.sample(n=N_MUESTRA, random_state=42)

assert len(df_muestra) == N_MUESTRA
assert "neighbourhood_cleansed" in df_muestra.columns

In [17]:
TARGET = "daily_price"

media_pop = df[TARGET].mean()
var_pop = df[TARGET].var()

media_mue = df_muestra[TARGET].mean()
var_mue = df_muestra[TARGET].var()

tabla_validacion = pd.DataFrame(
    {
        "Metrica": [
            "Media (€)",
            "Varianza",
            "Desvio Estándar (€)",
            "Mediana (€)",
        ],
        "Poblacion": [
            round(media_pop, 2),
            round(var_pop, 2),
            round(df[TARGET].std(), 2),
            round(df[TARGET].median(), 2),
        ],
        "Muestra (n=750)": [
            round(media_mue, 2),
            round(var_mue, 2),
            round(df_muestra[TARGET].std(), 2),
            round(df_muestra[TARGET].median(), 2),
        ],
        "Diferencia Relativa (%)": [
            round(abs(media_pop - media_mue) / media_pop * 100, 2),
            round(abs(var_pop - var_mue) / var_pop * 100, 2),
            round(
                abs(df[TARGET].std() - df_muestra[TARGET].std())
                / df[TARGET].std()
                * 100,
                2,
            ),
            round(
                abs(df[TARGET].median() - df_muestra[TARGET].median())
                / df[TARGET].median()
                * 100,
                2,
            ),
        ],
    }
)

display(tabla_validacion)

,Metrica,Poblacion,Muestra (n=750),Diferencia Relativa (%)
0,Media (€),103.74,99.55,4.03
1,Varianza,19031.98,8510.88,55.28
2,Desvio Estándar (€),137.96,92.25,33.13
3,Mediana (€),75.00,75.00,0.00


En esta ejecución, la media difiere 4.03 %, la mediana 0.00 % y la varianza 55.28 %. Las diferencias describen el centro y la dispersión de esta muestra.

In [ ]:
zona = "neighbourhood_cleansed"
proporciones = pd.DataFrame({
    "Base (%)": df[zona].value_counts(normalize=True) * 100,
    "Muestra (%)": df_muestra[zona].value_counts(normalize=True) * 100,
}).fillna(0).sort_index()
proporciones["Diferencia (p. p.)"] = (
    proporciones["Muestra (%)"] - proporciones["Base (%)"]
)
display(proporciones.round(2))

,Base (%),Muestra (%),Diferencia (p. p.)
neighbourhood_cleansed,,,
1,17.61,17.47,-0.15
2,12.75,12.80,0.05
3,11.74,11.73,-0.00
4,10.07,10.13,0.06
5,9.36,9.33,-0.03
6,12.36,12.40,0.04
7,7.13,7.20,0.07
8,9.78,9.73,-0.05
9,9.18,9.20,0.02


La mayor diferencia es de 0,15 puntos porcentuales y se observa en la zona 1.

La muestra regenerada con el procedimiento del notebook difiere del CSV histórico. Se guarda por separado para conservar ese archivo.

In [18]:
# Guardamos la muestra
df_muestra.to_csv(DATA_BASE_DIR / "milan_airbnb_muestra_regenerada.csv", index=False)

## Muestra estratificada después de aplicar los filtros y la selección de variables del EDA

Partimos del CSV completo, aplicamos los filtros del EDA y extraemos una segunda muestra de 750 anuncios por zona.

In [16]:
df_crudo = pd.read_csv(
    PROJECT_DIR / "data" / "raw" / "milan_airbnb.csv",
    index_col=0,
)

columnas_cruce = [
    "latitude", "longitude", "bathrooms", "bedrooms",
    "beds", "accommodates", "guests_included", "daily_price",
]

casos_eda = df_crudo.loc[[14, 7882], columnas_cruce]
claves_base = pd.MultiIndex.from_frame(df[columnas_cruce])
claves_eda = pd.MultiIndex.from_frame(casos_eda)

if any((claves_base == clave).sum() != 1 for clave in claves_eda):
    raise ValueError("Cada caso señalado en el EDA debe coincidir con una fila.")

df = df.loc[
    ~claves_base.isin(claves_eda)
    & df["guests_included"].le(df["accommodates"])
].copy()

# Aplicamos los descartes del EDA que todavía figuran en el CSV completo.
columnas_descartadas_eda = [
    "beds", "bed_type", "Heating", "Kitchen",
    "Wheelchair_accessible", "Suitable_for_events",
]
df = df.drop(columns=columnas_descartadas_eda)

In [17]:
N_MUESTRA = 750
grupos = df.groupby("neighbourhood_cleansed", sort=True)
cupos_proporcionales = grupos.size() * N_MUESTRA / len(df)
cupos = cupos_proporcionales.astype(int)

faltantes = N_MUESTRA - cupos.sum()
restos = (cupos_proporcionales - cupos).sort_values(
    ascending=False, kind="stable"
)
cupos.loc[restos.index[:faltantes]] += 1

rng = np.random.default_rng(42)
df_muestra = pd.concat([
    grupo.sample(n=int(cupos.loc[zona]), random_state=rng)
    for zona, grupo in grupos
])

assert len(df_muestra) == N_MUESTRA

In [18]:
TARGET = "daily_price"

media_pop = df[TARGET].mean()
var_pop = df[TARGET].var()

media_mue = df_muestra[TARGET].mean()
var_mue = df_muestra[TARGET].var()

tabla_validacion = pd.DataFrame(
    {
        "Metrica": [
            "Media (€)",
            "Varianza",
            "Desvio Estándar (€)",
            "Mediana (€)",
        ],
        "Poblacion": [
            round(media_pop, 2),
            round(var_pop, 2),
            round(df[TARGET].std(), 2),
            round(df[TARGET].median(), 2),
        ],
        "Muestra (n=750)": [
            round(media_mue, 2),
            round(var_mue, 2),
            round(df_muestra[TARGET].std(), 2),
            round(df_muestra[TARGET].median(), 2),
        ],
        "Diferencia Relativa (%)": [
            round(abs(media_pop - media_mue) / media_pop * 100, 2),
            round(abs(var_pop - var_mue) / var_pop * 100, 2),
            round(
                abs(df[TARGET].std() - df_muestra[TARGET].std())
                / df[TARGET].std()
                * 100,
                2,
            ),
            round(
                abs(df[TARGET].median() - df_muestra[TARGET].median())
                / df[TARGET].median()
                * 100,
                2,
            ),
        ],
    }
)

display(tabla_validacion)

,Metrica,Poblacion,Muestra (n=750),Diferencia Relativa (%)
0,Media (€),103.71,100.25,3.33
1,Varianza,19068.54,7538.18,60.47
2,Desvio Estándar (€),138.09,86.82,37.13
3,Mediana (€),75.00,75.00,0.00


La media del precio en la muestra difiere un **3.33 %** de la base filtrada y la mediana difiere un **0.00 %**. La varianza difiere un **60.47 %**. La estratificación mantiene las proporciones por zona, con el redondeo necesario para asignar 750 cupos. Las diferencias en el precio describen esta muestra en particular.

In [ ]:
zona = "neighbourhood_cleansed"
proporciones = pd.DataFrame({
    "Base (%)": df[zona].value_counts(normalize=True) * 100,
    "Muestra (%)": df_muestra[zona].value_counts(normalize=True) * 100,
}).fillna(0).sort_index()
proporciones["Diferencia (p. p.)"] = (
    proporciones["Muestra (%)"] - proporciones["Base (%)"]
)
display(proporciones.round(2))

,Base (%),Muestra (%),Diferencia (p. p.)
neighbourhood_cleansed,,,
1,17.62,17.60,-0.02
2,12.79,12.80,0.01
3,11.71,11.73,0.02
4,10.06,10.00,-0.06
5,9.37,9.33,-0.04
6,12.37,12.40,0.03
7,7.14,7.20,0.06
8,9.77,9.73,-0.03
9,9.18,9.20,0.02


La mayor diferencia es de 0,06 puntos porcentuales y se observa en la zona 7.

In [19]:
# Guardamos la base filtrada y la muestra
df.to_csv(DATA_BASE_DIR / "milan_airbnb_completo_filtrado.csv", index=False)
df_muestra.to_csv(DATA_BASE_DIR / "milan_airbnb_muestra_filtrada.csv", index=False)